# Football Analytics Engine — Private Data Loader

Notebook untuk Android/Google Colab. Mengambil **satu arsip musim per eksekusi** dari Football-Data.co.uk, menyimpannya di Google Drive pribadi, menggabungkan CSV lokal, lalu menjalankan walk-forward backtest.

**Penting:** gunakan hanya untuk analisis pribadi nonkomersial. Notebook membatasi unduhan otomatis maksimal **2 arsip dalam 7 hari berjalan** sesuai fair-use yang dipublikasikan sumber. Tidak ada scraping berulang, tidak ada upload CSV mentah ke GitHub, dan jangan membagikan folder Google Drive berisi data mentah.

Mulai dengan `1011` (musim 2010/11). Setelah berhasil, ubah `season_code` pada sel unduhan untuk musim lain. Format kode: `1011`, `1112`, ..., `2627`. Jalankan paling banyak dua unduhan per tujuh hari.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys

ROOT = Path('/content/drive/MyDrive/FootballAnalyticsEngine')
DATA_DIR = ROOT / 'football-data'
ARCHIVE_DIR = ROOT / 'archives'
OUTPUT_DIR = ROOT / 'results'
for folder in (DATA_DIR, ARCHIVE_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

REPO = Path('/content/football-analytics-engine')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/alvian092/football-analytics-engine.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')], check=True)
sys.path.insert(0, str(REPO))
print('Setup selesai. Data mentah disimpan di Google Drive pribadi.')


## 1. Ambil satu arsip musim

Ubah hanya `season_code` untuk memilih musim. Sel ini tidak mengunduh banyak musim sekaligus. Jika arsip musim tersebut sudah ada di Drive, unduhan dilewati.


In [ ]:
import csv, re, zipfile, io, requests
from datetime import datetime, timezone, timedelta

season_code = '1011'  # Contoh: 1011=2010/11, 2627=2026/27
if not re.fullmatch(r'\d{4}', season_code):
    raise ValueError('Kode musim harus empat digit, misalnya 1011.')

archive_path = ARCHIVE_DIR / f'{season_code}.zip'
season_dir = DATA_DIR / season_code
season_dir.mkdir(parents=True, exist_ok=True)
log_path = ROOT / 'download_log.csv'

if archive_path.exists() and any(season_dir.glob('*.csv')):
    print(f'Arsip {season_code} sudah tersedia. Tidak mengunduh ulang.')
else:
    now = datetime.now(timezone.utc)
    recent = []
    if log_path.exists():
        with log_path.open(newline='', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                try:
                    stamp = datetime.fromisoformat(row['downloaded_at'])
                    if stamp.tzinfo is None:
                        stamp = stamp.replace(tzinfo=timezone.utc)
                    if now - stamp <= timedelta(days=7):
                        recent.append(row)
                except Exception:
                    pass
    if len(recent) >= 2:
        raise RuntimeError('Batas fair-use tercapai: sudah ada 2 unduhan dalam 7 hari berjalan. Tunggu sampai salah satunya lebih dari 7 hari.')

    url = f'https://football-data.co.uk/mmz4281/{season_code}/data.zip'
    print('Mengambil satu arsip:', url)
    response = requests.get(url, timeout=90, headers={'User-Agent': 'FootballAnalyticsEngine-personal-research/1.0'})
    response.raise_for_status()
    if not response.content.startswith(b'PK'):
        raise RuntimeError('Respons sumber bukan file ZIP yang valid; arsip tidak disimpan.')
    archive_path.write_bytes(response.content)

    extracted = 0
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        for info in zf.infolist():
            if info.is_dir() or not info.filename.lower().endswith('.csv'):
                continue
            safe_name = Path(info.filename).name
            if not safe_name:
                continue
            target = season_dir / safe_name
            target.write_bytes(zf.read(info))
            extracted += 1
    if extracted == 0:
        archive_path.unlink(missing_ok=True)
        raise RuntimeError('Tidak ditemukan CSV di dalam ZIP.')

    new_file = not log_path.exists()
    with log_path.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['downloaded_at', 'season_code', 'url'])
        if new_file:
            writer.writeheader()
        writer.writerow({'downloaded_at': now.isoformat(), 'season_code': season_code, 'url': url})
    print(f'Berhasil menyimpan {extracted} CSV untuk musim {season_code} di Google Drive.')


## 2. Normalisasi semua musim yang sudah tersimpan

Sel ini membaca CSV dari Google Drive pribadi. File mentah tidak dikirim ke GitHub.


In [ ]:
from engine.football_data import import_football_data_directory

matches = import_football_data_directory(DATA_DIR)
normalized_path = OUTPUT_DIR / 'matches.csv'
matches.to_csv(normalized_path, index=False)
print('Pertandingan selesai:', len(matches))
print('Kompetisi:', matches['competition'].nunique())
print('Musim:', ', '.join(sorted(matches['season'].dropna().astype(str).unique())))
print('Rentang tanggal:', matches['date'].min(), 'sampai', matches['date'].max())
print('File hasil:', normalized_path)


## 3. Jalankan walk-forward backtest

Model hanya menggunakan pertandingan yang sudah terjadi sebelum tanggal prediksi. Hasil ini evaluasi probabilitas, bukan jaminan keuntungan. Dengan data yang masih sedikit, banyak pertandingan akan dilewati karena riwayat belum cukup.


In [ ]:
backtest_dir = OUTPUT_DIR / 'backtest'
cmd = [sys.executable, str(REPO / 'run_backtest.py'), '--input', str(normalized_path), '--output-dir', str(backtest_dir)]
completed = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print('LOG:', completed.stderr[-4000:])
print('Exit code:', completed.returncode)

metrics_file = backtest_dir / 'metrics.json'
predictions_file = backtest_dir / 'predictions.csv'
if metrics_file.exists():
    print('\nMETRICS JSON:')
    print(metrics_file.read_text(encoding='utf-8'))
print('\nHasil disimpan di Google Drive:', backtest_dir)
print('Jangan unggah atau publikasikan CSV mentah sumber.')
